<a href="https://colab.research.google.com/github/htf100/transfomer/blob/main/docs_nnx/chinese_aomen_double_cls.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!cp -r "/content/drive/MyDrive/chinese_common" "/content/"

In [3]:
!pip install lightning


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.0/853.0 kB 63.4 MB/s eta 0:00:00


In [5]:
"""训练：python dict_double.py；只拆分：python dict_double.py split；预测：脚本后加评论。"""

import csv
import json
import sys
from pathlib import Path

import lightning.pytorch as pl
import torch
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split

# 首次拆分时使用 SOURCE_PATH；已有拆分文件时可直接复用。
ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
SOURCE_PATH = Path("/content/chinese_common/weibo_senti_100k.csv")
OUTPUT_DIR = ROOT / "dict_double_results"
SPLIT_DIR = OUTPUT_DIR / "splits"
TRAIN_PATH = SPLIT_DIR / "train.csv"
VAL_PATH = SPLIT_DIR / "valid.csv"
TEST_PATH = SPLIT_DIR / "test.csv"
CHECKPOINT = OUTPUT_DIR / "review_double.ckpt"
MAX_LENGTH, HIDDEN_SIZE = 128, 128
num_heads = 4
d_head = HIDDEN_SIZE//num_heads  # 注意力头数固定为 4，hidden_size 必须能被 4 整除。
BATCH_SIZE, EPOCHS = 32, 20  # 固定跑满 20 轮，每轮验证一次。
LR, SEED = 1e-3, 42
USE_GPU = torch.cuda.is_available()  # 有 CUDA 就使用 GPU；改为 False 可强制 CPU。
GPU_ID = 0                         # 使用第几张可见的 GPU，从 0 开始。
UNK, PAD, CLS = 0, 1, 2
B = BATCH_SIZE
L = MAX_LENGTH


def read_reviews(path):
    """自动识别 label,text 或 label,review 表头；0 是负面，1 是正面。"""
    with open(path, encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file, strict=True)
        fields = reader.fieldnames or []
        text_key = next((key for key in ("text", "review") if key in fields), None)
        if "label" not in fields or text_key is None:
            raise ValueError(f"{path}：需要 label 列，以及 text 或 review 文本列")
        samples = [
            ((row.get(text_key) or "").replace("\ufeff", "").strip(), int(row["label"]))
            for row in reader
        ]
    if not samples or any(not text or label not in (0, 1) for text, label in samples):
        raise ValueError(f"{path}：需要非空评论和 0/1 标签")
    return samples


def split_dataset(source_path=None, output_dir=None):
    """清理重复和冲突标签后，用 PyTorch 按每类 80%/10%/10% 随机拆分。"""
    source_path = Path(SOURCE_PATH if source_path is None else source_path).expanduser()
    output_dir = Path(SPLIT_DIR if output_dir is None else output_dir).expanduser()
    samples = read_reviews(source_path)
    labels_by_text = {}
    for text, label in samples:
        labels_by_text.setdefault(text, set()).add(label)
    conflicts = {text for text, labels in labels_by_text.items() if len(labels) > 1}
    clean = [(text, next(iter(labels))) for text, labels in labels_by_text.items() if len(labels) == 1]
    conflict_rows = sum(text in conflicts for text, _ in samples)

    generator = torch.Generator().manual_seed(SEED)
    splits = {name: [] for name in ("train", "valid", "test")}
    for label in (0, 1):
        group = [sample for sample in clean if sample[1] == label]
        if len(group) < 10:
            raise ValueError(f"清理后标签 {label} 至少需要 10 条，才能拆成三个非空集合")
        train_size, val_size = round(len(group) * 0.8), round(len(group) * 0.1)
        parts = random_split(
            group, [train_size, val_size, len(group) - train_size - val_size],
            generator=generator,
        )
        for bucket, part in zip(splits.values(), parts):
            bucket.extend(part)

    report = {
        "source": str(source_path.resolve()), "seed": SEED, "ratios": [0.8, 0.1, 0.1],
        "original_samples": len(samples), "conflicting_texts": len(conflicts),
        "removed_conflicting_rows": conflict_rows,
        "removed_same_label_duplicates": len(samples) - conflict_rows - len(clean),
        "clean_samples": len(clean), "splits": {},
    }
    output_dir.mkdir(parents=True, exist_ok=True)
    for name, subset in splits.items():
        order = torch.randperm(len(subset), generator=generator).tolist()
        path = output_dir / f"{name}.csv"
        with path.open("w", encoding="utf-8", newline="") as file:
            writer = csv.writer(file)
            writer.writerow(["label", "text"])
            writer.writerows((subset[index][1], subset[index][0]) for index in order)
        counts = {str(label): sum(y == label for _, y in subset) for label in (0, 1)}
        report["splits"][name] = {"samples": len(subset), "labels": counts}
        print(f"{name}: {len(subset)} 条 | 负面 {counts['0']}，正面 {counts['1']} | {path}")
    (output_dir / "split_summary.json").write_text(
        json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8",
    )
    print(f"原始 {len(samples)} 条 → 清理后 {len(clean)} 条；冲突文本 {len(conflicts)} 个")
    return report


def ensure_split_dataset():
    """复用本项目已有的三个拆分文件；缺少文件时才清洗原始数据。"""
    paths = (TRAIN_PATH, VAL_PATH, TEST_PATH)
    if all(path.is_file() and path.stat().st_size > 0 for path in paths):
        print(f"已找到清洗后的训练、验证和测试集，跳过拆分：{SPLIT_DIR}")
        return
    print(f"拆分文件不完整，开始从原始数据清洗并生成：{SPLIT_DIR}")
    split_dataset()


def encode(text, vocab, max_length):
    text = text.strip()
    if not text:
        raise ValueError("评论不能为空")
    if vocab.get("[CLS]") != CLS:
        raise ValueError("词表缺少 [CLS]，或 [CLS] 的 ID 与当前模型不一致；请重新训练模型")
    tokens = [CLS] + [vocab.get(char, UNK) for char in text[:max_length - 1]]  # 留一个位置给 CLS
    return tokens + [PAD] * (max_length - len(tokens))


def make_loader(samples, vocab, max_length, shuffle=False):
    tokens = torch.tensor([encode(text, vocab, max_length) for text, _ in samples])
    labels = torch.tensor([label for _, label in samples])
    return DataLoader(
        TensorDataset(tokens, labels), batch_size=BATCH_SIZE,
        shuffle=shuffle, pin_memory=USE_GPU, drop_last=False,
    )


class encoderBlock(nn.Module):
    def __init__(self, hidden_size=128):#num_heads、d_head在前面定义了个全局变量 现在在这并没有被引用
        super().__init__()
        self.linear_q = nn.Linear(hidden_size, hidden_size)
        self.linear_k = nn.Linear(hidden_size, hidden_size)
        self.linear_v = nn.Linear(hidden_size, hidden_size)

        self.dropout_0 = nn.Dropout(0.1)
        self.layer_norm_0 = nn.LayerNorm(hidden_size)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_size, hidden_size * 2), nn.ReLU(),
            nn.Linear(hidden_size * 2, hidden_size),
        )
        self.layer_norm_1 = nn.LayerNorm(hidden_size)
        self.dropout_1 = nn.Dropout(0.1)
        # self.classifier = nn.Linear(hidden_size, 2)

    def forward(self, tokens, mask):
        dropout_p = 0.5 if self.training else 0.0
        B, L, _ = tokens.size()
        x = tokens
        # mask = tokens != PAD                              # [B, L]，True 是真实字符。此时token是整数id 还未尽过embedding进行位置编码 编码后变成[B, L, hidden_size]，随后拆成多个头。
        q = self.linear_q(x)                # [B, L, hidden_size]，随后拆成多个头。
        k = self.linear_k(x)
        v = self.linear_v(x)
        q = q.view(B, L, num_heads, d_head).transpose(1, 2)  # [B, H, L, d_head]
        k = k.view(B, L, num_heads, d_head).transpose(1, 2)
        v = v.view(B, L, num_heads, d_head).transpose(1, 2)
        attended = F.scaled_dot_product_attention(
            q, k, v, attn_mask=mask[:, None, None, :], dropout_p=dropout_p,
        )
        attended = attended.transpose(1, 2).contiguous().view(B, L, num_heads * d_head)    #把张量变成“连续存储”的形式

        x = self.layer_norm_0(x + self.dropout_0(attended))
        x = self.layer_norm_1(x + self.dropout_1(self.ffn(x)))
        return x  # 取 [CLS]；PAD 已在注意力中被 mask 排除。  B*L*H



class ReviewModel(pl.LightningModule):
    def __init__(self, vocab, max_length=128, hidden_size=128, lr=1e-3, num_layers=2):
        super().__init__()
        self.save_hyperparameters()  # 词表和结构参数随模型保存，预测时自动恢复。
        self.loss_history = []  # 每轮的编号、训练 loss 和验证 loss。
        self.embedding = nn.Embedding(len(vocab), hidden_size, padding_idx=PAD)
        self.position_embedding = nn.Embedding(max_length, hidden_size)

        self.layers = nn.ModuleList([
            encoderBlock(hidden_size)
            for _ in range(num_layers)])
        self.classifier = nn.Linear(hidden_size, 2)

    def forward(self, tokens):
        mask = tokens != PAD                              # [B, L]，True 是真实字符。此时token是整数id 还未尽过embedding进行位置编码 编码后变成[B, L, hidden_size]，随后拆成多个头。
        positions = torch.arange(tokens.size(1), device=tokens.device)
        x = self.embedding(tokens) + self.position_embedding(positions)     #[B, L, hidden_size]，随后拆成多个头。
        for layer in self.layers:
            x = layer(x, mask)
        return self.classifier(x[:, 0, :])  # 取 [CLS]；PAD 已在注意力中被 mask 排除。

    def shared_step(self, batch, stage):
        tokens, labels = batch
        logits = self(tokens)
        loss = F.cross_entropy(logits, labels)
        accuracy = (logits.argmax(1) == labels).float().mean()
        self.log_dict(
            {f"{stage}_loss": loss, f"{stage}_acc": accuracy},
            on_step=False, on_epoch=True, prog_bar=True, batch_size=len(labels),
        )
        return loss

    def training_step(self, batch, batch_idx):
        return self.shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        self.shared_step(batch, "val")

    def test_step(self, batch, batch_idx):
        self.shared_step(batch, "test")

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.hparams.lr)

    def on_train_epoch_end(self):
        metrics = self.trainer.callback_metrics
        self.loss_history.append((
            self.current_epoch + 1, metrics["train_loss"].item(), metrics["val_loss"].item(),
        ))
        print(
            f"Epoch {self.current_epoch + 1}/{self.trainer.max_epochs} | "
            f"train loss={metrics['train_loss']:.4f}, acc={metrics['train_acc']:.2%} | "
            f"val loss={metrics['val_loss']:.4f}, acc={metrics['val_acc']:.2%}"
        )


def plot_losses(history, path):
    """保存整次训练的损失曲线；无需图形界面，适用于 GPU 服务器。"""
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    from matplotlib.figure import Figure
    from matplotlib.ticker import MaxNLocator

    epochs, train_losses, val_losses = zip(*history)
    figure = Figure(figsize=(8, 4.5), layout="constrained")
    FigureCanvasAgg(figure)
    axes = figure.subplots()
    axes.plot(epochs, train_losses, marker="o", markersize=3, label="Train loss")
    axes.plot(epochs, val_losses, marker="o", markersize=3, label="Validation loss")
    axes.set(xlabel="Epoch", ylabel="Cross-entropy loss", title="Training and validation loss")
    axes.xaxis.set_major_locator(MaxNLocator(integer=True))
    axes.set_ylim(bottom=0)
    axes.grid(alpha=0.25)
    axes.legend()
    figure.savefig(path, dpi=180)
    print(f"损失曲线：{path}")


@torch.inference_mode()
def predict(model, texts):
    model.eval()
    for text in texts:
        tokens = encode(text, model.hparams.vocab, model.hparams.max_length)
        inputs = torch.tensor([tokens], device=model.device)
        probabilities = model(inputs).softmax(dim=-1)[0]
        label = probabilities.argmax().item()
        print(f"{text} → {['负面', '正面'][label]}（模型概率 {probabilities[label].item():.1%}）")
        text_limit = model.hparams.max_length - 1  # 第 0 位留给 [CLS]。
        if len(text.strip()) > text_limit:
            print(f"  仅分析前 {text_limit} 个字符")


def train():
    pl.seed_everything(SEED, workers=True)
    ensure_split_dataset()
    training = read_reviews(TRAIN_PATH)
    validation = read_reviews(VAL_PATH)
    vocab = {"[UNK]": UNK, "[PAD]": PAD,"[CLS]": CLS}
    for text, _ in training:  # 只用训练集建词表，验证和测试中的新字符映射为 UNK。
        for char in text:
            vocab.setdefault(char, len(vocab))
    train_loader = make_loader(training, vocab, MAX_LENGTH, shuffle=True)
    val_loader = make_loader(validation, vocab, MAX_LENGTH)
    print(f"训练 {len(training)} 条 | 验证 {len(validation)} 条 | 词表 {len(vocab)}")
    print(f"训练设备：{f'cuda:{GPU_ID}' if USE_GPU else 'cpu'}")

    checkpoint = ModelCheckpoint(
        dirpath=CHECKPOINT.parent, filename=CHECKPOINT.stem,
        monitor="val_loss", mode="min", save_top_k=1, enable_version_counter=False,
    )
    # GPU 训练：Lightning 自动把模型和每个 batch 搬到指定显卡，并管理反向传播。
    trainer = pl.Trainer(
        accelerator="gpu" if USE_GPU else "cpu",
        devices=[GPU_ID] if USE_GPU else 1,
        max_epochs=EPOCHS, num_sanity_val_steps=0, enable_progress_bar=False,
        logger=CSVLogger(OUTPUT_DIR / "training_logs", name="weibo"),
        callbacks=[checkpoint],
    )
    model = ReviewModel(vocab, MAX_LENGTH, HIDDEN_SIZE, LR)
    trainer.fit(model, train_loader, val_loader)
    plot_losses(model.loss_history, OUTPUT_DIR / "loss_curve.png")
    best = ReviewModel.load_from_checkpoint(
        checkpoint.best_model_path, map_location="cpu", weights_only=True,
    )
    best_epoch = torch.load(checkpoint.best_model_path, map_location="cpu", weights_only=True)["epoch"] + 1
    print(f"最佳模型：第 {best_epoch} 轮 | {checkpoint.best_model_path}")
    # 训练完成后才评估测试集；测试集不参与模型选择。
    testing = read_reviews(TEST_PATH)
    trainer.test(best, dataloaders=make_loader(testing, vocab, MAX_LENGTH))
    return best


if __name__ == "__main__":
    args = sys.argv[1:] if "__file__" in globals() else []
    if args == ["split"]:
        ensure_split_dataset()
    elif args:
        model = ReviewModel.load_from_checkpoint(CHECKPOINT, map_location="cpu", weights_only=True)
        model.to(f"cuda:{GPU_ID}" if USE_GPU else "cpu")
        predict(model, args)
    else:
        model = train()
        predict(model, ["服务很好，下次还来", "等了很久，体验很差"])


INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


已找到清洗后的训练、验证和测试集，跳过拆分：/content/dict_double_results/splits


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /content/dict_double_results exists and is not empty.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.py

训练 93536 条 | 验证 11692 条 | 词表 5819
训练设备：cuda:0


┏━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name               ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ embedding          │ Embedding  │  744 K │ train │     0 │
│ 1 │ position_embedding │ Embedding  │ 16.4 K │ train │     0 │
│ 2 │ layers             │ ModuleList │  231 K │ train │     0 │
│ 3 │ classifier         │ Linear     │    258 │ train │     0 │
└───┴────────────────────┴────────────┴────────┴───────┴───────┘

...

INFO: `Trainer.fit` stopped: `max_epochs=20` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=20` reached.


Epoch 20/20 | train loss=0.0222, acc=99.19% | val loss=0.1243, acc=97.90%
损失曲线：/content/dict_double_results/loss_curve.png
最佳模型：第 4 轮 | /content/dict_double_results/review_double.ckpt


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.9826377034187317     │
│         test_loss         │    0.05050390213727951    │
└───────────────────────────┴───────────────────────────┘

服务很好，下次还来 → 正面（模型概率 100.0%）
等了很久，体验很差 → 正面（模型概率 100.0%）
